In [1]:
# 09_sun_abraham_comparison.ipynb
# Agreement across modern difference-in-differences estimators. We estimate the
# dynamic effect of the acute health shock on CHE with three estimators -- two-way
# fixed effects, Callaway-Sant'Anna, and the Sun-Abraham interaction-weighted
# (saturated) event study -- and compare their event-time profiles and overall ATT.
# Figure is grayscale, no titles, 600-dpi PNG and PDF; tables to ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
from differences import ATTgt
import pyfixest as pf

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
for th in ("che25", "che40"):
    p[th + "f"] = p[th].astype(float)

# ---- TWFE event study and static ATT ----
def twfe(df, yn):
    d = df.dropna(subset=[yn]).copy()
    d["k"] = np.where(d["cohort"] > 0, d["year"] - d["cohort"], np.nan)
    d["kb"] = d["k"].apply(lambda k: np.nan if pd.isna(k) else max(-4, min(4, int(k))))
    evk = [-4, -3, -2, 0, 1, 2, 3, 4]
    for kk in evk:
        d[f"e{kk}"] = ((d["cohort"] > 0) & (d["kb"] == kk)).astype(int)
    cnt = d.groupby("PIDWON")["year"].transform("size")
    es = d[cnt >= 2].set_index(["PIDWON", "year"])
    cols = [f"e{kk}" for kk in evk]
    r = PanelOLS(es[yn], es[cols], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
    ev = {kk: (r.params[f"e{kk}"], r.std_errors[f"e{kk}"]) for kk in evk}; ev[-1] = (0.0, 0.0)
    # static ATT
    d["post"] = ((d["cohort"] > 0) & (d["year"] >= d["cohort"])).astype(int)
    es2 = d.set_index(["PIDWON", "year"])
    rs = PanelOLS(es2[yn], es2[["post"]], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
    return ev, (rs.params["post"], rs.std_errors["post"])

# ---- Callaway-Sant'Anna event + overall ----
def cs(df, yn):
    d = df[["PIDWON", "year", "cohort", yn]].dropna(subset=[yn]).copy()
    d["cohort"] = d["cohort"].replace(0, np.nan); d = d.set_index(["PIDWON", "year"])
    a = ATTgt(data=d, cohort_column="cohort", base_period="universal")
    a.fit(formula=yn, est_method="dr", control_group="not_yet_treated", progress_bar=False)
    e = a.aggregate("event"); e.columns = [str(c) for c in e.columns]
    ev = {int(k): (float(r.iloc[0]), float(r.iloc[1]) if not pd.isna(r.iloc[1]) else 0.0) for k, r in e.iterrows()}
    s = a.aggregate("simple"); s.columns = [str(c) for c in s.columns]
    return ev, (float(s.iloc[0, 0]), float(s.iloc[0, 1]))

# ---- Sun-Abraham (saturated) event + overall ----
def sunab(df, yn):
    d = df.dropna(subset=[yn])
    m = pf.event_study(data=d, yname=yn, idname="PIDWON", tname="year", gname="cohort",
                       estimator="saturated", att=False, cluster="PIDWON")
    agg = m.aggregate()
    ev = {int(k): (float(agg.loc[k, "Estimate"]), float(agg.loc[k, "Std. Error"])) for k in agg.index}
    # overall ATT: observation-weighted average of post periods
    dd = d[d.cohort > 0].copy(); dd["k"] = dd["year"] - dd["cohort"]
    w = dd[dd["k"] >= 0].groupby("k").size()
    ks = [k for k in ev if k >= 0]
    num = sum(ev[k][0] * w.get(k, 0) for k in ks); den = sum(w.get(k, 0) for k in ks)
    return ev, (num / den, np.nan)

# ---- run for both thresholds ----
rows = []
events = {}
for yn, lab in [("che25f", "CHE 25%"), ("che40f", "CHE 40%")]:
    tw_ev, tw_att = twfe(p, yn)
    cs_ev, cs_att = cs(p, yn)
    sa_ev, sa_att = sunab(p, yn)
    events[yn] = {"twfe": tw_ev, "cs": cs_ev, "sa": sa_ev}
    rows.append({"outcome": lab, "twfe_att": tw_att[0], "twfe_se": tw_att[1],
                 "cs_att": cs_att[0], "cs_se": cs_att[1], "sa_att": sa_att[0]})
comp = pd.DataFrame(rows); comp.to_csv(f"{TAB}/t09_estimator_comparison.csv", index=False)

# ---- figure: CHE40 event-time, three estimators ----
ev = events["che40f"]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.axhline(0, color="0.5", lw=0.8); ax.axvline(-0.5, color="0.7", ls="--", lw=1)
specs = [("twfe", "Two-way FE", "s", "0.62", "--", -0.12),
         ("cs", "Callaway--Sant'Anna", "o", "0.12", "-", 0.0),
         ("sa", "Sun--Abraham", "^", "0.4", "-.", 0.12)]
for key, labl, mk, col, ls, off in specs:
    d = ev[key]; ks = sorted(d.keys())
    xs = [k + off for k in ks]; ys = [d[k][0] for k in ks]; es = [1.96 * d[k][1] for k in ks]
    ax.errorbar(xs, ys, yerr=es, fmt=mk + ls, color=col, ms=5, lw=1.5, capsize=2.5,
                label=labl, markerfacecolor="white", markeredgewidth=1.3)
ax.set_xlabel("Years relative to first health shock (k)")
ax.set_ylabel("Effect on P(CHE 40%) vs k$=-1$")
ax.legend(frameon=False, fontsize=9.5)
save_fig(fig, "fig12_estimator_comparison")

print("Overall ATT by estimator:")
print(comp.round(4).to_string(index=False))


Overall ATT by estimator:
outcome  twfe_att  twfe_se  cs_att  cs_se  sa_att
CHE 25%    0.0797   0.0045  0.0688 0.0042  0.0749
CHE 40%    0.0404   0.0032  0.0361 0.0029  0.0383
